
# Byte‑Pair Encoding (BPE) From Scratch  
This notebook builds **a BPE tokenizer from scratch** and walks through the same four iterations shown in the following diagram.

<div style="width:500px;margin:auto;">

<img src="../img/bpe.png" alt="seg embedding" width="750"/>

</div>

* **Corpus:** `"AACGCACTATATA"` (spaces added for readability)  
* **Algorithm:**  
  1. Start with a vocabulary of individual characters.  
  2. **Count** all bigrams (adjacent symbol pairs) in the corpus.  
  3. **Merge** the most‑frequent bigram into a new symbol.  
  4. Repeat steps 2–3 until the desired vocabulary size is reached.

The implementation follows the seminal paper by Sennrich *et al.* (2016) and the Hugging Face LLM‑course notes.


In [1]:
from collections import Counter
from typing import List, Tuple

def get_stats(corpus: List[List[str]]) -> Counter:
    """Count frequency of adjacent symbol pairs in the corpus."""
    pairs = Counter()
    for word in corpus:
        for i in range(len(word)-1):
            pairs[(word[i], word[i+1])] += 1
    return pairs

def merge_pair(pair: Tuple[str, str], corpus: List[List[str]]) -> List[List[str]]:
    """Merge the given pair everywhere in the corpus."""
    merged = []
    bigram = ''.join(pair)
    for word in corpus:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word)-1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        merged.append(new_word)
    return merged

def bpe(corpus: List[str], num_merges: int = 10):
    """Perform BPE merges and return the transformed corpus & final vocab."""
    corpus_sym = [list(word) for word in corpus]
    vocab = set().union(*corpus_sym)
    for _ in range(num_merges):
        stats = get_stats(corpus_sym)
        if not stats:
            break
        best = stats.most_common(1)[0][0]
        corpus_sym = merge_pair(best, corpus_sym)
        vocab.add(''.join(best))
    return corpus_sym, sorted(vocab)

In [2]:
corpus = ["AACGCACTATATA"]
print('Original corpus:', corpus)

new_corpus, final_vocab = bpe(corpus, num_merges=3)

print('\nAfter 3 merges:', new_corpus)
print('Final vocabulary:', final_vocab)

Original corpus: ['AACGCACTATATA']

After 3 merges: [['A', 'AC', 'G', 'C', 'AC', 'TATA', 'TA']]
Final vocabulary: ['A', 'AC', 'C', 'G', 'T', 'TA', 'TATA']


In [3]:
def run_verbose(corpus: str, iterations: int = 3):
    corpus_sym = [list(corpus)]
    vocab = set(corpus_sym[0])
    for it in range(iterations+1):
        print(f'\nIteration {it}')
        print('Corpus :', ' '.join(corpus_sym[0]))
        print('Vocab  :', sorted(vocab))
        if it == iterations:
            break
        stats = get_stats(corpus_sym)
        best = stats.most_common(1)[0][0]
        print('Most frequent pair:', best)
        corpus_sym = merge_pair(best, corpus_sym)
        vocab.add(''.join(best))

run_verbose("AACGCACTATATA", iterations=3)


Iteration 0
Corpus : A A C G C A C T A T A T A
Vocab  : ['A', 'C', 'G', 'T']
Most frequent pair: ('T', 'A')

Iteration 1
Corpus : A A C G C A C TA TA TA
Vocab  : ['A', 'C', 'G', 'T', 'TA']
Most frequent pair: ('A', 'C')

Iteration 2
Corpus : A AC G C AC TA TA TA
Vocab  : ['A', 'AC', 'C', 'G', 'T', 'TA']
Most frequent pair: ('TA', 'TA')

Iteration 3
Corpus : A AC G C AC TATA TA
Vocab  : ['A', 'AC', 'C', 'G', 'T', 'TA', 'TATA']


In [11]:
english_text = "Twinkle, twinkle, little star"

run_verbose(english_text, iterations=15)


Iteration 0
Corpus : T w i n k l e ,   t w i n k l e ,   l i t t l e   s t a r
Vocab  : [' ', ',', 'T', 'a', 'e', 'i', 'k', 'l', 'n', 'r', 's', 't', 'w']
Most frequent pair: ('l', 'e')

Iteration 1
Corpus : T w i n k le ,   t w i n k le ,   l i t t le   s t a r
Vocab  : [' ', ',', 'T', 'a', 'e', 'i', 'k', 'l', 'le', 'n', 'r', 's', 't', 'w']
Most frequent pair: ('w', 'i')

Iteration 2
Corpus : T wi n k le ,   t wi n k le ,   l i t t le   s t a r
Vocab  : [' ', ',', 'T', 'a', 'e', 'i', 'k', 'l', 'le', 'n', 'r', 's', 't', 'w', 'wi']
Most frequent pair: ('wi', 'n')

Iteration 3
Corpus : T win k le ,   t win k le ,   l i t t le   s t a r
Vocab  : [' ', ',', 'T', 'a', 'e', 'i', 'k', 'l', 'le', 'n', 'r', 's', 't', 'w', 'wi', 'win']
Most frequent pair: ('win', 'k')

Iteration 4
Corpus : T wink le ,   t wink le ,   l i t t le   s t a r
Vocab  : [' ', ',', 'T', 'a', 'e', 'i', 'k', 'l', 'le', 'n', 'r', 's', 't', 'w', 'wi', 'win', 'wink']
Most frequent pair: ('wink', 'le')

Iteration 5
Corpus : T

In [14]:
chinese_text = "鹅，鹅，鹅，曲项向天歌。"

run_verbose(chinese_text, iterations=8)


Iteration 0
Corpus : 鹅 ， 鹅 ， 鹅 ， 曲 项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '，']
Most frequent pair: ('鹅', '，')

Iteration 1
Corpus : 鹅， 鹅， 鹅， 曲 项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '，']
Most frequent pair: ('鹅，', '鹅，')

Iteration 2
Corpus : 鹅，鹅， 鹅， 曲 项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '鹅，鹅，', '，']
Most frequent pair: ('鹅，鹅，', '鹅，')

Iteration 3
Corpus : 鹅，鹅，鹅， 曲 项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '鹅，鹅，', '鹅，鹅，鹅，', '，']
Most frequent pair: ('鹅，鹅，鹅，', '曲')

Iteration 4
Corpus : 鹅，鹅，鹅，曲 项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '鹅，鹅，', '鹅，鹅，鹅，', '鹅，鹅，鹅，曲', '，']
Most frequent pair: ('鹅，鹅，鹅，曲', '项')

Iteration 5
Corpus : 鹅，鹅，鹅，曲项 向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '鹅，鹅，', '鹅，鹅，鹅，', '鹅，鹅，鹅，曲', '鹅，鹅，鹅，曲项', '，']
Most frequent pair: ('鹅，鹅，鹅，曲项', '向')

Iteration 6
Corpus : 鹅，鹅，鹅，曲项向 天 歌 。
Vocab  : ['。', '向', '天', '曲', '歌', '项', '鹅', '鹅，', '鹅，鹅，', '鹅，鹅，鹅，', '鹅，鹅，鹅